# Article 02 — The RDA FAIR Maturity Model In Depth
## A Complete Guide to All 41 Indicators

**Series:** FAIR Data Maturity Framework  
**Author:** Ali Shahmohammadi, Ph.D. — Takeda Pharmaceutical  
**Prerequisites:** Article 01 — Introduction to FAIR Data Principles

---

## Learning Objectives

1. Understand the structure and purpose of the RDA FAIR Maturity Model
2. Navigate all 41 indicators across F, A, I, R
3. Apply the Essential / Important / Useful priority framework
4. Understand what each indicator is testing and what evidence satisfies it
5. Identify the most critical indicators for a pharmaceutical R&D dataset

## 1. What is the RDA FAIR Maturity Model?

The **RDA FAIR Data Maturity Model** was published in 2020 by the  
[RDA FAIR Data Maturity Model Working Group](https://doi.org/10.15497/rda00050).

It provides:
- **41 measurable indicators** derived from the 15 FAIR sub-principles
- **5-level compliance scoring** per indicator: Not Applicable / Not Assessed / Not Compliant / Partially Compliant / Compliant
- **3 priority levels**: Essential (must meet), Important (should meet), Useful (nice to have)
- **Two scopes per indicator**: Metadata (M) and/or Data (D)

### Why 41 indicators, not 15?

The 15 FAIR sub-principles are high-level and abstract. The RDA Working Group operationalised them  
by breaking each into concrete, measurable criteria. For example, F1 ("Globally unique and persistent  
identifier") becomes 4 indicators:
- F1-01M: Metadata has a persistent identifier
- F1-01D: Data has a persistent identifier
- F1-02M: Metadata identifier is globally unique
- F1-02D: Data identifier is globally unique

This granularity enables precise gap analysis and automated assessment.

In [ ]:
# Load the full RDA indicator catalogue
from fair_toolkit import RDA_INDICATORS, EvidenceLevel, FAIRPrinciple, IndicatorScope
import pandas as pd

# Build a summary DataFrame
records = []
for ind in RDA_INDICATORS:
    records.append({
        "ID": ind.id,
        "Principle": ind.principle.value,
        "Sub-principle": ind.sub_principle.value,
        "Scope": ind.scope.value,
        "Priority": ind.priority.value,
        "Name": ind.name[:60] + "..." if len(ind.name) > 60 else ind.name,
    })

df = pd.DataFrame(records)
print(f"Total indicators: {len(df)}")
print()
print(df.groupby(["Principle", "Priority"])["ID"].count().unstack(fill_value=0))

In [ ]:
# Display all F indicators with full details
from rich.table import Table
from rich.console import Console
from fair_toolkit import get_indicators_by_principle, FAIRPrinciple

console = Console()

for principle in [FAIRPrinciple.F, FAIRPrinciple.A, FAIRPrinciple.I, FAIRPrinciple.R]:
    indicators = get_indicators_by_principle(principle)

    table = Table(
        title=f"[{principle.value}] — {len(indicators)} indicators",
        show_lines=True
    )
    table.add_column("ID", style="bold cyan", width=16)
    table.add_column("Priority", width=10)
    table.add_column("Scope", width=6)
    table.add_column("Name", width=55)

    priority_colours = {"essential": "red", "important": "yellow", "useful": "green"}
    for ind in indicators:
        colour = priority_colours[ind.priority.value]
        table.add_row(
            ind.id,
            f"[{colour}]{ind.priority.value}[/{colour}]",
            ind.scope.value,
            ind.name,
        )
    console.print(table)
    console.print()

## 2. Deep Dive: The Findable Indicators (F1–F4)

### F1 — Globally Unique and Persistent Identifiers

This is **the most foundational** FAIR requirement. Without identifiers, nothing else works:
- You can't link metadata to data (F3)
- Search engines can't resolve references (F4)
- Other datasets can't cite your data (I3)

**What counts as a Persistent Identifier (PID)?**

| PID Type | Example | Use case |
|---------|---------|---------|
| **DOI** (Digital Object Identifier) | `10.5281/zenodo.7654321` | Publications, datasets, code |
| **ARK** (Archival Resource Key) | `ark:/13030/tf5p30086k` | Long-term archives, libraries |
| **Handle** | `20.500.12345/abc123` | Institutional repositories |
| **IGSN** | `IGSN:AU1234567` | Physical samples, geological specimens |
| **ORCID** | `0000-0002-1825-0097` | Researcher identifiers |
| **ROR** | `https://ror.org/02mhbdp94` | Organisation identifiers |

**What does NOT count?**
- Internal LIMS IDs (not globally unique, not persistent)
- URL-only identifiers (URLs break, DOIs persist)
- UUIDs without a registered resolver

### F2 — Rich Metadata

Rich metadata answers: *"What is this data? Should I use it for my purpose?"*

Minimum metadata for a pharmaceutical dataset:
1. **Title** — descriptive, not just "Dataset 1"
2. **Creator** — with ORCID
3. **Date created/modified**
4. **Data type** — with ontology term (OBI, EDAM)
5. **Organism/cell line** — with NCBITaxon/Cellosaurus ID
6. **Assay method** — with OBI term
7. **Keywords** — using MeSH or domain ontology terms
8. **Abstract/description** — including known limitations
9. **Data quality indicators** — Z-prime, signal:noise, batch effects
10. **Related publication** — with DOI

In [ ]:
# Examine the F indicators in detail
from fair_toolkit import get_indicator

findable_ids = [ind.id for ind in RDA_INDICATORS if ind.principle.value == "F"]
print(f"Findable indicators ({len(findable_ids)} total)\n")

for iid in findable_ids:
    ind = get_indicator(iid)
    print(f"{'─'*70}")
    print(f"  {ind.id}  [{ind.priority.value.upper()}] [{ind.scope.value}]")
    print(f"  {ind.name}")
    print(f"  Question: {ind.question}")
    if ind.example:
        print(f"  Pharma example: {ind.example}")
    print()

## 3. Scoring the 41 Indicators

Each indicator is scored on a 5-level scale:

| Score | Meaning | Numeric weight |
|-------|---------|----------------|
| **Not Applicable** | Indicator does not apply to this data type | — |
| **Not Assessed** | Not yet evaluated | — |
| **Not Compliant** | Criterion is not met | 0.0 |
| **Partially Compliant** | Criterion is partially met | 0.5 |
| **Compliant** | Criterion is fully met | 1.0 |

### Computing a FAIR Score

The overall FAIR score is computed as:

$$
\text{FAIR Score} = \frac{1}{4} \sum_{d \in \{F,A,I,R\}} \frac{\sum_{i \in d} w_i}{|d_{assessed}|}
$$

where $w_i$ is the numeric weight of indicator $i$'s compliance level  
and $|d_{assessed}|$ is the count of assessed (non-NA) indicators in dimension $d$.

### Priority-Weighted Assessment

The RDA model recommends tracking **Essential indicator compliance separately**  
because a dataset can score 80% overall but have 0% on Essential indicators  
— which would mean it is fundamentally not FAIR.

A dataset is considered **minimally FAIR** when:
- All **Essential** indicators are at least Partially Compliant
- At least 50% of **Important** indicators are Compliant

In [ ]:
# Demonstrate the scoring system with a synthetic example
from fair_toolkit import ManualFAIRAssessor, ComplianceScore, EvidenceLevel

# Create an assessor for a hypothetical internal pharma dataset
assessor = ManualFAIRAssessor(
    dataset_id="DS-EXAMPLE-001",
    dataset_title="CAR-T Viability Assay — Internal Dataset",
    assessed_by="Ali Shahmohammadi",
    notes="Synthetic example for illustration — not a real assessment",
)

# Score the Findable indicators
assessor.score("RDA-F1-01D", ComplianceScore.NOT_COMPLIANT,
               evidence="Only internal LIMS ID, no global PID")
assessor.score("RDA-F1-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="Metadata stored in LIMS only, no external PID")
assessor.score("RDA-F1-02D", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-F1-02M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-F2-01M", ComplianceScore.PARTIALLY_COMPLIANT,
               evidence="Title, date, creator captured; assay conditions missing")
assessor.score("RDA-F3-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="No explicit metadata→data identifier link")
assessor.score("RDA-F4-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="Not registered in any external catalogue")

# Score the Accessible indicators
assessor.score("RDA-A1-01M", ComplianceScore.PARTIALLY_COMPLIANT,
               evidence="Access process exists but is informal (email the data owner)")
assessor.score("RDA-A1-02M", ComplianceScore.NOT_COMPLIANT,
               evidence="LIMS API exists but not documented publicly")
assessor.score("RDA-A1-03D", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-A1-04M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-A1.1-01M", ComplianceScore.NOT_APPLICABLE,
               reason="Not yet accessible via any external protocol")
assessor.score("RDA-A1.1-01D", ComplianceScore.NOT_APPLICABLE)
assessor.score("RDA-A1.2-01M", ComplianceScore.COMPLIANT,
               evidence="Enterprise SSO (Azure AD) controls access")
assessor.score("RDA-A1.2-01D", ComplianceScore.COMPLIANT,
               evidence="Azure AD OAuth 2.0")
assessor.score("RDA-A2-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="Metadata only in LIMS — no external metadata deposit")

# Score Interoperable
assessor.score("RDA-I1-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="Metadata in proprietary LIMS XML, no public schema")
assessor.score("RDA-I1-01D", ComplianceScore.PARTIALLY_COMPLIANT,
               evidence="Data exported as CSV — open format but no schema")
assessor.score("RDA-I1-02M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I1-02D", ComplianceScore.PARTIALLY_COMPLIANT)
assessor.score("RDA-I2-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="Using internal codebook, not OBO ontologies")
assessor.score("RDA-I2-01D", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I3-01M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I3-01D", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I3-02M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I3-02D", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I3-03M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-I3-04M", ComplianceScore.NOT_COMPLIANT)

# Score Reusable
assessor.score("RDA-R1-01M", ComplianceScore.PARTIALLY_COMPLIANT,
               evidence="Some key metadata fields populated but incomplete")
assessor.score("RDA-R1.1-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="No licence specified — default 'all rights reserved'")
assessor.score("RDA-R1.1-02M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-R1.1-03M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-R1.2-01M", ComplianceScore.PARTIALLY_COMPLIANT,
               evidence="ELN audit trail exists (ALCOA+) but not in PROV-O format")
assessor.score("RDA-R1.2-02M", ComplianceScore.COMPLIANT,
               evidence="Benchling ELN with GxP audit trail — ALCOA+ compliant")
assessor.score("RDA-R1.3-01M", ComplianceScore.NOT_COMPLIANT,
               evidence="No community metadata standard applied")
assessor.score("RDA-R1.3-01D", ComplianceScore.PARTIALLY_COMPLIANT,
               evidence="CSV format with README — open but no community standard validation")
assessor.score("RDA-R1.3-02M", ComplianceScore.NOT_COMPLIANT)
assessor.score("RDA-R1.3-02D", ComplianceScore.NOT_COMPLIANT)

# Print the scorecard
assessor.print_scorecard()
print()
print("Progress:", assessor.progress())

## 4. Priority-Based Gap Analysis

Not all 41 indicators are equally critical. The RDA model assigns three priority levels:

### Essential (must be met for minimum FAIRness)
These indicators represent the absolute foundation. A dataset failing Essential indicators  
cannot be considered FAIR regardless of its score on other indicators.

**Key Essential indicators for pharma:**
- F1-01D, F1-01M — PIDs on data and metadata
- F2-01M — Rich metadata
- F3-01M — Metadata explicitly links to data
- F4-01M — Metadata is indexed/harvestable
- A1-02M — Metadata accessible via open API
- A2-01M — Metadata persists when data is gone
- R1-01M — Rich descriptive attributes for reuse
- R1.1-01M — Explicit licence
- R1.3-01M — Metadata meets community standard

### Important (should be met for meaningful FAIRness)
Meeting these indicators moves a dataset from 'barely FAIR' to 'meaningfully FAIR'.

### Useful (nice to have)
These represent best practices that improve FAIRness at the margins.

In [ ]:
# Show Essential gap analysis from our example
from fair_toolkit import EvidenceLevel

result = assessor.build_result()

print("=== FAIR Dimension Scores ===")
for dim, score in result.dimension_scores.items():
    bar = "█" * int(score / 5) + "░" * (20 - int(score / 5))
    print(f"  {dim:8s}: {bar}  {score}%")

print()
print("=== Essential Indicator Gaps ===")
essential_gaps = result.get_gaps(EvidenceLevel.ESSENTIAL)
for gap in essential_gaps:
    print(f"  ✗ [{gap.principle.value}] {gap.indicator_id} — {gap.indicator_name}")
    if gap.evidence:
        print(f"      Evidence: {gap.evidence}")

print(f"\n  Total essential gaps: {len(essential_gaps)}")
print(f"  Overall FAIR score: {result.overall_score}%")

## Summary

The RDA FAIR Maturity Model provides the most complete and authoritative framework  
for measuring FAIR data compliance. Key takeaways:

1. **41 indicators** derived from the 15 FAIR sub-principles
2. **3 priority levels** — focus on Essential gaps first
3. **5-level scoring** — Not Applicable / Not Assessed / Not Compliant / Partially Compliant / Compliant
4. **Dual scope** — most indicators apply to both Metadata (M) and Data (D)
5. The `ManualFAIRAssessor` in this toolkit handles the complete assessment workflow

**Typical FAIR scores for pharma internal data:**
- Internal LIMS dataset (no external sharing): ~15–30% overall
- Published dataset in domain repository: ~60–75% overall
- Best-practice FAIR dataset (e.g., ChEMBL): ~85–95% overall

---

**Next Article:** [Article 03 — The Pistoia Alliance FAIR Maturity Matrix](./03_pistoia_fair_maturity_matrix.ipynb)